## Summary 

This notebook takes the milk recordings and fits the MilkBot lactation curve model using the lactation curve package. The fitted characteristics are saved to the blob and used downstream.

# Bi-modal milk flow project, calculate lactation curve characteristics 

Notebook contains the following
- part 2 of data preprocessing farm data
- Calculation of lactation curve characteristics 
- Calculation of proportion of bi-modal-milk flow (DME)

Calculated characteristics are written to the blob as it takes a lot of memmory and time to calculate them for each lactation.

Import packages

lactation curve package: van Leerdam, M. B., de Kok, D., Osei-Tete, J. A., & Hostens, M. (2026). Bovi-analytics/bovi lactationcurve: v.1.1.6 (Version 1.1.6) [Computer software]. https://doi.org/10.5281/zenodo.18715144

In [0]:
%pip install lactationcurve

from lactationcurve.characteristics import calculate_characteristic
from lactationcurve.fitting import fit_lactation_curve

In [0]:
#try out if the import of the function went alright 

dim = [10,20,30]
y = [20,35,40]

print(calculate_characteristic(dim, y, model = 'MilkBot', characteristic = 'time_to_peak'))
print(calculate_characteristic(
        dim,
        y,
        model = 'wood',
        characteristic = 'persistency',
        fitting='frequentist',
        persistency_method = 'derived'))

In [0]:
#other packages
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
from pyspark.sql.window import Window
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
# from statsmodels.formula.api import ols
from scipy.stats import norm 
# import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

Connect data

In [0]:
#read dataframe from the US blob 
parquet_file_path = "wasbs://bovianalytics-data@bovianalyticsstorage.blob.core.windows.net/us/SunnySide/milk_flow_data/"

# Read the Parquet file into a DataFrame > now set to a limit to first make the code working
bi_modal_df = spark.read.parquet(parquet_file_path)


# Data preperation

In [0]:
#get rid of yesterday in the column name as it is no longer relevant and causes confusion
bi_modal_df = bi_modal_df.toDF(*[
    c.replace('yesterday', '').replace('Yesterday', '') 
    for c in bi_modal_df.columns
])

#drop two rows where DIM = 0 
bi_modal_df = bi_modal_df.filter(bi_modal_df['DaysInMilk'].isNotNull())

# Calculate CalvingDate and CalvingSeason for each lactation
bi_modal_df = (
    bi_modal_df
    .withColumn(
        "LastMilkDate",
        F.to_timestamp("LastMilkTime", "M/d/yyyy h:mm a")
    )
    .withColumn(
        "CalvingDate",
        F.date_sub(F.col("LastMilkDate"), F.col("DaysInMilk"))
    )
    .withColumn(
        "CalvingSeason",
        F.when(F.month("CalvingDate").isin(12, 1, 2), "Winter")
         .when(F.month("CalvingDate").isin(3, 4, 5), "Spring")
         .when(F.month("CalvingDate").isin(6, 7, 8), "Summer")
         .otherwise("Autumn")
    )
    .withColumn(
        'CalvingYear',
        F.year("CalvingDate")
    )
)

#somehow there has been an introduction of dulpicate rows, erase them again 
bi_modal_df = bi_modal_df.dropDuplicates()



In [0]:
#create criteria to filter lactations so we get less weird lactation characteristics 

# They must have minimal 10 measurements 
# -----------------------------
# 1. Count measurements per lactation
# -----------------------------
counts = (
    bi_modal_df.groupBy("LactationId")
      .agg(F.count("*").alias("n_rows"))
      .filter(F.col("n_rows") >= 10)
)

# -----------------------------
# 2. Check presence of early and late DIM measurements 
# There must be a milkrecording in the first 50 days and in the last 50 days of the lactation 
# -----------------------------
dim_flags = (
    bi_modal_df.groupBy("LactationId")
      .agg(
          F.max(F.col("DaysInMilk") <= 50).cast("int").alias("has_early"),
          F.max((F.col("DaysInMilk") >= 255) & (F.col("DaysInMilk") <= 305))
            .cast("int").alias("has_late")
      )
      .filter("has_early = 1 AND has_late = 1")
)

# -----------------------------
# 3. Keep only lactations that satisfy ALL criteria
# -----------------------------
valid_lactations = (
    counts.join(dim_flags, on="LactationId", how="inner")
          .select("LactationId")
)

filtered_df = (
    bi_modal_df.join(valid_lactations, on="LactationId", how="inner")
)

# display(filtered_df)

In [0]:
#Descriptive stats df
number_of_recordings = bi_modal_df.count()
number_of_animals = bi_modal_df.select('AnimalNumber').distinct().count()
number_of_parities = bi_modal_df.select('LactationNumber').distinct().count()
number_of_lactations = bi_modal_df.select('LactationId').distinct().count()
print(f'Number of recordings: {number_of_recordings}')
print(f'Number of animals: {number_of_animals}')
print(f'Number of parities: {number_of_parities}')
print(f'Number of lactations: {number_of_lactations}')
print(f'Average amount of lactations per animal: {number_of_lactations/number_of_animals}')

In [0]:
#Descriptive stats df
number_of_recordings = filtered_df.count()
number_of_animals = filtered_df.select('AnimalNumber').distinct().count()
number_of_parities = filtered_df.select('LactationNumber').distinct().count()
number_of_lactations = filtered_df.select('LactationId').distinct().count()
print(f'Number of recordings: {number_of_recordings}')
print(f'Number of animals: {number_of_animals}')
print(f'Number of parities: {number_of_parities}')
print(f'Number of lactations: {number_of_lactations}')
print(f'Average amount of lactations per animal: {number_of_lactations/number_of_animals}')

## Seperate Bi-midal from non-bi-modal 


In [0]:
#create a variable for proportion of DME per lactation 
#create a window per lactation 
w = Window.partitionBy("LactationId")

#Number of milkings with observations (assuming no missing milk measurements)
filtered_df = filtered_df.withColumn(
    "Valid_sessions",
    F.lit(3)
)

# with missing milk measurements
# subset = subset.withColumn(
#     "Valid_sessions",
#     F.col("Flow30_60Session1Yesterday").isNotNull().cast("int") +
#     F.col("Flow30_60Session2Yesterday").isNotNull().cast("int") +
#     F.col("Flow30_60Session3Yesterday").isNotNull().cast("int")
# )

#Number of milkings with bimodal milk flow curve 
filtered_df = filtered_df.withColumn(
    "BIM_sessions",
    F.col("BIM_1").cast("int") +
    F.col("BIM_2").cast("int") +
    F.col("BIM_3").cast("int")
)

filtered_df = filtered_df.withColumn(
    "BIM_prop",
    F.sum("BIM_sessions").over(w) / F.sum("Valid_sessions").over(w)
)

#rename column BIM to DME 
filtered_df = filtered_df.withColumnRenamed('BIM_prop', 'DME_prop')
filtered_df = filtered_df.withColumnRenamed('BIM_prop_logit', 'DME_prop_logit')

#split based on a threshold 
bins = [0, 0.20, 0.40, 1.0]
labels = ["low", "middle", "high"]

filtered_df = filtered_df.withColumn(
    "DME_prop_cat",
    F.when(F.col("DME_prop") <= bins[1], labels[0])
     .when((F.col("DME_prop") > bins[1]) & (F.col("DME_prop") <= bins[2]), labels[1])
     .when((F.col("DME_prop") > bins[2]) & (F.col("DME_prop") <= bins[3]), labels[2])
)

In [0]:
#alternative create a histplot as the visualisation of databricks somethimes dissapoints 

data = filtered_df.toPandas()

# data['BIM_prop'].plot.hist(bins=100, title="BIM proportion per lactation")
plt.show()


In [0]:
# Add the square root column to the pandas DataFrame
data['SqrtBMIProp'] = data['BIM_prop'].apply(lambda x: np.sqrt(x))

plt.hist(
    data["SqrtBMIProp"].dropna(),
    bins=100
)
plt.title("sqrt(BIM proportion) per lactation")
plt.show()

# Only apply log to positive values to avoid -inf and NaN
data['logBMIProp'] = data['BIM_prop'].apply(
    lambda x: np.log(x) if x > 0 else 0
)

plt.hist(
    data["logBMIProp"].dropna(),
    bins=100
)
plt.title("log(BIM proportion) per lactation")
plt.show()

eps = 1e-6

data["BIM_prop_logit"] = np.log(
    (data["BIM_prop"] + eps) / (1 - data["BIM_prop"] + eps)
)
plt.hist(
    data["BIM_prop_logit"],
    bins=100
)
plt.title("logit(BIM proportion) per lactation")
plt.show()

# Arcsine-sqrt transform
data["BIM_prop_arcsine"] = np.arcsin(np.sqrt(data["BIM_prop"]))

plt.hist(
    data["BIM_prop_arcsine"],
    bins=100
)
plt.title("arcsine(BIM proportion) per lactation")
plt.show()

In [0]:
#calculate mean and 95% c.i. of BIM prop

mean = data['BIM_prop'].mean()
sem = data['BIM_prop'].std(ddof=1) / np.sqrt(len(data['BIM_prop']))

ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=len(data['BIM_prop'])-1,
    loc=mean,
    scale=sem
)

print('The mean BIM proportion is: ', mean)
print('The standard error of the mean is: ', sem)
print('The 95% confidence interval is: ', ci_low, ci_high)

## Calculate lactationcurve characteristic 

In [0]:
#Use the lactationcurve characteristic part of the lc package to calculate LC characteristics 
#first convert df to pandas
# use subset for now
bim_pd = data

time_to_peak_list = []
lactation_id = []
peak_yield_list = []
cummulative_yield_list = []
persistency_list = []


#create a loop to calculate all the characteristics for each LactationId
for lac_id in bim_pd['LactationId'].unique():
    dim = bim_pd[bim_pd['LactationId'] == lac_id]['DaysInMilk'].values
    milk_yield = bim_pd[bim_pd['LactationId'] == lac_id]['DailyMilkYield'].values
    try:
        time_to_peak = calculate_characteristic(
            dim,
            milk_yield,
            model='milkbot',
            characteristic='time_to_peak',
            fitting='frequentist'
        )
    except Exception:
        time_to_peak = None
    peak_yield = calculate_characteristic(
        dim,
        milk_yield,
        model = 'milkbot',
        characteristic = 'peak_yield',
        fitting='frequentist'
        )
    cummulative_yield = calculate_characteristic(
        dim,
        milk_yield,
        model = 'milkbot',
        characteristic = 'cumulative_milk_yield',
        fitting='frequentist'
        )
    persistency = calculate_characteristic(
        dim,
        milk_yield,
        model = 'milkbot',
        characteristic = 'persistency',
        fitting='frequentist',
        persistency_method = 'literature'
        )
    time_to_peak_list.append(time_to_peak)
    lactation_id.append(lac_id)
    peak_yield_list.append(peak_yield)
    cummulative_yield_list.append(cummulative_yield)
    persistency_list.append(persistency)

# Create dataframe 

characteristics = pd.DataFrame(
    list(zip(lactation_id, time_to_peak_list, peak_yield_list, cummulative_yield_list, persistency_list)),
    columns=['LactationId', 'TimeToPeak', 'PeakYield', 'CummulativeYield', 'Persistency']
)

#join the dataframes together 
df_final = bim_pd.merge(characteristics, on='LactationId', how='right')

#group by lactation id 
df_final = df_final.groupby('LactationId', as_index=False).first()


In [0]:
#drop all rows with a null for time to peak and a negative cummulative milk yield 
df_cleaned = df_final[
    df_final["TimeToPeak"].notna() & 
    (df_final["TimeToPeak"] < 305) & #You cannot peak after you finished your lactation 
    # (df_final["Persistency"] < 10000) & #gut feeling 
    (df_final['PeakYield'] < 127) #world record milk kg per day 
]


In [0]:
#convert df to spark 
df_final_spark = spark.createDataFrame(df_cleaned)

#write to the blob 
lactation_curve_characteristics = df_final_spark.write.mode("overwrite"). \
    parquet(
        "wasbs://bovianalytics-data@bovianalyticsstorage.blob.core.windows.net/us/SunnySide/lactation_curve_characteristics_half_time_persistency"
    )

In [0]:
#visualize model variables 
plt.hist(
    df_final['TimeToPeak'],
    bins=100
)
plt.title('Time to peak distribution')
plt.xlabel('Time to peak')
plt.ylabel('Frequency')
plt.show()

plt.hist(df_final['PeakYield'], bins=100)
plt.title('Peak distribution')
plt.xlabel('Peak')
plt.ylabel('Frequency')
plt.show()

plt.hist(df_final['CummulativeYield'], bins=100)
plt.title('Cummulative yield distribution')
plt.xlabel('Cummulative yield')
plt.ylabel('Frequency')
plt.show()

plt.hist(df_final['Persistency'], bins=100)
plt.title('Persistency distribution')
plt.xlabel('Persistency')
plt.ylabel('Frequency')
plt.show()

plt.hist(df_final['DailyMilkYield'], bins = 100)
plt.title('Daily milk yield distribution')
plt.xlabel('Daily milk yield')
plt.ylabel('Frequency')
plt.show

## Create lactationcurve visualisations of bimodality 

In [0]:
#rename column BIM to DME 
data.rename(columns={'BIM_prop': 'DME_prop'}, inplace=True)
data.rename(columns ={'BIM_prop_logit': 'DME_prop_logit'}, inplace=True)


#split based on a threshold 
bins = [0, 0.20, 0.40, 1.0]
labels = ["low", "middle", "high"]

data["DME_prop_cat"] = pd.cut(data["DME_prop"], bins=bins, labels=labels, include_lowest=True)

#convert lactationnumber to 4 categories, 1,2,3 where everything above 3 becomes 3. 
data["LactationCat"] = data["LactationNumber"].clip(upper=3)

#groupby lactationId to get one observation per lactation
observations = data.groupby('LactationId').first()

In [0]:
#alternative create a categorial variable out of BMI prop 

# Create a plot to visualize the number of lactations per category
counts = observations['DME_prop_cat'].value_counts().reindex(["low", "middle", "high"])
plt.bar(counts.index, counts.values, color=[plt.cm.Reds(0.4), plt.cm.Reds(0.7), plt.cm.Reds(1.0)])

# Print the column name at the middle of each bar
for i, (cat, count) in enumerate(counts.items()):
    plt.text(i, count / 2, str(count), ha='center', va='center', fontsize=10, color='black')

plt.title('DME proportion distribution')
plt.xlabel('DME proportion')
plt.ylabel('Frequency')
plt.show()



### Number of obeservations per Parity and DME category

In [0]:

#count number of LactationCat 
counts = observations['LactationCat'].value_counts().reindex([1, 2, 3])
plt.bar(counts.index, counts.values, color=[plt.cm.Greens(0.4), plt.cm.Greens(0.7), plt.cm.Greens(1.0)])

# Print the column name at the middle of each bar
for i, (cat, count) in enumerate(counts.items()):
    plt.text(i, count / 2, str(count), ha='center', va='center', fontsize=10, color='black')

plt.title('Lactation number distribution')
plt.xlabel('Lactation number')
plt.ylabel('Frequency')
plt.show()

In [0]:
#make overview of number of lactations per parity per categroy

for parity in observations['LactationCat'].unique():
    counts = observations[observations['LactationCat'] == parity]['DME_prop_cat'].value_counts()
    print(f"Parity {parity}:")
    print(counts)
    print("\n")


### fit a lactationcurve per DME category 

In [0]:
#add lactationcat column
filtered_df = filtered_df.withColumn('LactationCat', F.when(filtered_df['LactationNumber'] > 3, 3).otherwise(filtered_df['LactationNumber']))



#filter out the lactations with a bad LC fit
#list comes from notebook df_cleaned
lactations_to_exclude = [2866, 3488, 3089, 2924, 3028, 3042, 3105, 4126, 4713, 4452, 3133, 3072, 3387, 3173, 3239, 3240, 4720, 5282, 3244, 3276, 3495, 3140, 3721, 5129, 3752, 3798, 3282, 3060, 4775, 4877, 3557, 2867, 3067, 5246, 3269, 3324, 3458, 3249, 4405, 3401, 3330, 4712, 5048, 4648, 4793, 4518, 3426, 5283, 3374, 5271, 3971, 4345, 4210, 4054, 4105, 2918, 4892, 2940, 2884, 5278, 2881, 2951, 2966, 5033, 4687, 5305, 4168, 5172, 2986, 3318, 4554, 5267, 3882, 3973, 5197, 4678, 3869, 3466, 3820, 5340, 5384, 3268, 3082, 4170, 3532, 4963, 4942, 4954, 5012, 5229, 3203, 4378, 5070, 3680, 3643, 3355, 4769, 4509, 5079, 3186, 3224, 3321, 4724, 4636, 3524, 3853, 4707, 4171, 3368, 4797, 5803, 5614, 6361, 7530, 5740, 6301, 6082, 5571, 7118, 7550, 5845, 6476, 6011, 6040, 6509, 5480, 5892, 7197, 5905, 6422, 6273, 6337, 5786, 5655, 7160, 7234, 6706, 6234, 7230, 6194, 5612, 6375, 5676, 7349, 6050, 6275, 6655, 6756, 5937, 6091, 5507, 5466, 5716, 6127, 6261, 7181, 7372, 7420, 6249, 6141, 5728, 6798, 5667, 7348, 6166, 6335, 7270, 7012, 6311, 6215, 7423, 6029, 5945, 7363, 5659, 5709, 5966, 5999, 6075, 6521, 5503, 5795, 5556, 6137, 6343, 6212, 6363, 5602, 6402, 7150, 7292, 5799, 6713, 7052, 7492, 5853, 6184, 5922, 5988, 6897, 5725, 6206, 5942, 5689, 5613, 5849, 7553, 5958, 6749, 6549, 7358, 5416, 6292, 6815, 7547, 6753, 6808, 6834, 6048, 6341, 7273, 7403, 7421, 5570, 7300, 7407, 5719, 5654, 6242, 7445, 6238, 7458, 5712, 6114, 5634, 5986, 6220, 6251, 7471, 6777, 7496, 5648, 5873, 7415, 5537, 7135, 6033, 6278, 5911, 1270, 1699, 883, 2027, 613, 642, 1198, 2200, 772, 1869, 210, 1892, 2683, 1135, 2738, 1689, 548, 2022, 2013, 758, 1939, 618, 1382, 1427, 2256, 348, 2383, 572, 2520, 925, 2292, 1026, 1113, 1818, 1146, 1251, 1360, 1711, 1739, 1375, 1632, 2494, 1240, 207, 2556, 1966, 594, 568, 1451, 537, 823, 1737, 2163, 7, 514, 1431, 1498, 2136, 2214, 728, 434, 1963, 2119, 2333, 129, 928, 1872, 1134, 2800, 1638, 2512, 395, 1609, 589, 527, 1333, 1320, 2452, 288, 945, 1705, 1003, 1897, 62, 125, 1666, 2541, 1179, 1337, 2300, 2615, 1792, 752, 990, 1053, 1937, 2152, 141, 2592, 1511, 997, 303, 1744, 188, 1860, 2264, 1437, 2005, 892, 2162, 124, 302, 652, 1202, 2423, 1584, 716, 1862, 745, 1361, 2626, 2673, 841, 1107, 1217, 2208, 8638, 9454, 9558, 9403, 7987, 9324, 7865, 9416, 9207, 8127, 8915, 9161, 7962, 9716, 8141, 8290, 7556, 8255, 8306, 9178, 9653, 8005, 8399, 9732, 9533, 9291, 9624, 8987, 9476, 7660, 8185, 9298, 9833, 8615, 8201, 9079, 9854, 7581, 9816, 9008, 9637, 7703, 9844, 8653, 9202, 9655, 9690, 9822, 7891, 9300, 9513, 8002, 7628, 9371, 9760, 9832, 9186, 9753, 9066, 9204, 9269, 8839, 9845, 9807, 8119, 9056, 9256, 7650, 8868, 9717, 7947, 9831, 9871, 7620, 8061, 9536, 9872, 9320, 9502, 9819, 9880, 8783, 9567, 9201, 8635, 8959, 9463, 9005, 9142, 9486, 8468, 9837, 8633, 8642, 9024, 8118, 9835, 9248, 8099, 8063, 9656, 8732, 8541, 9552, 9572, 9260, 9223, 8045, 9243, 9356, 8103, 9808, 9503, 9773, 9266, 9438, 8618, 9599, 7999, 9067, 12471, 11520, 12207, 11459, 12436, 11954, 11686, 11456, 12509, 11950, 11461, 11350, 11491, 11607, 12533, 11445, 12035, 12440, 12063, 12442, 12075, 12346, 11947, 12106, 11553, 11347, 12409, 11557, 11399, 12291, 12587, 11771, 11752, 12457, 11715, 11412, 11990, 11354, 11315, 12028, 11376, 12573, 11884, 12303, 12153, 11836, 12152, 12625, 11410, 11528, 11734, 11922, 11628, 11592, 12570, 11351, 11653, 11751, 12403, 11834, 12501, 11727, 12484, 12387, 11451, 11889, 12552, 12619, 11737, 12089, 11993, 12164, 12074, 12205, 11280, 10032, 10013, 11024, 9898, 11072, 10651, 10116, 10252, 10950, 10010, 10539, 10135, 11120, 10006, 10683, 10079, 11252, 10002, 10583, 10131, 11180, 10376, 10216, 10833, 10737, 9937, 11011, 10347, 10569, 10807, 10104, 10899, 10963, 9903, 9953, 10101, 10767, 11232, 11107, 10447, 10056, 10083, 10905, 9930, 10578, 10685, 10940, 10538, 9992, 11123, 10333, 11082, 10799, 10100, 10142, 10365, 10172, 10657, 10976, 10003, 10475, 9927, 10743, 10995, 11061, 10134, 10024, 10946, 11020, 10317, 10381, 10859, 11007, 10496, 10619, 9989, 10004, 10068, 10093, 10105, 11281, 10402, 10452, 11032, 10440, 10086, 11066, 9887, 10067, 10710, 11238, 9913, 10308, 10454, 10152, 10355, 10065, 10753, 10955, 10830, 10023, 11167, 11015, 10825, 11299, 10097, 10593, 10055, 10188, 10865, 10148, 11186, 10372, 10051, 10270, 10543, 10552, 10516, 11109, 10679, 10884, 10256, 9922, 10491, 9986, 9999, 10185, 10843, 9996, 10429, 10655, 10670, 11197, 10513, 10515, 10693, 9897, 10156, 10929, 11166, 11230, 10867, 14832, 14536, 14997, 14752, 14874, 14477, 15161, 15270, 15904, 14935, 15716, 14558, 15234, 14783, 15203, 14813, 15188, 15849, 17386, 14860, 15543, 15330, 14774, 14534, 14656, 14861, 15560, 14804, 17326, 17276, 15076, 15704, 17283, 15812, 15896, 14735, 17357, 14637, 17342, 14510, 15832, 15018, 15218, 17427, 15415, 15567, 17323, 15443, 14872, 14791, 15125, 14526, 14550, 14672, 14987, 17487, 14607, 15213, 15089, 15743, 15536, 14901, 17406, 15396, 15461, 14506, 15054, 15295, 15521, 14844, 15048, 14701, 14754, 14763, 14939, 15000, 15629, 14544, 14609, 14552, 15045, 17352, 17418, 15284, 14893, 15316, 14911, 17367, 14674, 14583, 15017, 15744, 15816, 14693, 15430, 17277, 17497, 12782, 13104, 14439, 13344, 14327, 13965, 14112, 14013, 12972, 14095, 14470, 12762, 13471, 12877, 14131, 13914, 13388, 12751, 13884, 14334, 13028, 12742, 13330, 13940, 14360, 13012, 14440, 14457, 13881, 14222, 13826, 13806, 12758, 13396, 13367, 13934, 14094, 14259, 12760, 14329, 13502, 13777, 12816, 13796, 14285, 13393, 13519, 13585, 14090, 12932, 13573, 13645, 14110, 13284, 12793, 13798]

filtered_df = filtered_df.filter(~filtered_df['LactationId'].isin(lactations_to_exclude))

In [0]:
#convert to pandas for lc modelling
lc_df_panda = filtered_df.toPandas()

fit_lc = {}

for cat in lc_df_panda['DME_prop_cat'].unique():
    subset = lc_df_panda[lc_df_panda['DME_prop_cat'] == cat]
    dim = subset['DaysInMilk']
    my = subset["DailyMilkYield"]
    fit = fit_lactation_curve(dim, my, model = 'milkbot', fitting = 'frequentist')
    
    fit_lc[cat] = fit

#plot 
cat_order = ["high", "middle", "low"]
legend_labels = {
    "high": "high (>40%)",
    "middle": "middle (20-40%)",
    "low": "low (<20%)"
}
red_palette = [plt.cm.Reds(1.0), plt.cm.Reds(0.7), plt.cm.Reds(0.4)]
for i, cat in enumerate(cat_order):
    if cat in fit_lc:
        plt.plot(fit_lc[cat], label=legend_labels[cat], color=red_palette[i])
plt.legend()
plt.title('Lactation curve per DME proportion category')
plt.xlabel('Days in milk')
plt.ylabel('Milk yield (kg/day)')
plt.show()
 
 

### plot LC per parity per DME 

In [0]:
red_palette = [plt.cm.Reds(1.0), plt.cm.Reds(0.7), plt.cm.Reds(0.4)]
cat_order = ["high", "middle", "low"]

for lac_cat in lc_df_panda['LactationCat'].unique():
    subset_lac = lc_df_panda[lc_df_panda['LactationCat'] == lac_cat]
    fit_lc = {}

    for cat in cat_order:
        if cat in subset_lac['DME_prop_cat'].unique():
            subset = subset_lac[subset_lac['DME_prop_cat'] == cat]
            dim = subset['DaysInMilk']
            my = subset["DailyMilkYield"]
            fit = fit_lactation_curve(dim, my, model='milkbot', fitting='frequentist')
            fit_lc[cat] = fit

    for i, cat in enumerate(cat_order):
        if cat in fit_lc:
            plt.plot(fit_lc[cat], label=cat, color=red_palette[i])
    plt.legend()
    if lac_cat == 3:
        plt.title(f'Lactation curve per DME proportion category for parity {lac_cat}+')
    else:
        plt.title(f'Lactation curve per DME proportion category for parity {lac_cat}')
    plt.xlabel('Days in milk')
    plt.ylabel('Milk yield (kg/day)')
    plt.show()

## changes in DME over time 

In [0]:
#cut-off at dim 305 
data_305 = data[data['DaysInMilk'] < 305]

#alternative if you drop day 1
data_2_to_305 = data[data['DaysInMilk'] > 1]

grouped = data_305.groupby('DaysInMilk')[['BIM_sessions', 'Valid_sessions']].sum()
prop_DME_per_day = grouped['BIM_sessions'] / grouped['Valid_sessions']

prop_DME_per_day.plot()
plt.title('Proportion of milkings with DME per day')
plt.xlabel('Days in milk')
plt.ylabel('Proportion of DME')
plt.show()

In [0]:
#cut-off at dim 305 
data_305 = data[data['DaysInMilk'] < 305]

#alternative if you drop day 1
data_2_to_305 = data_305[data_305['DaysInMilk'] > 1]

grouped = data_2_to_305.groupby('DaysInMilk')[['BIM_sessions', 'Valid_sessions']].sum()
prop_DME_per_day = grouped['BIM_sessions'] / grouped['Valid_sessions']

prop_DME_per_day.plot()
plt.title('Proportion of milkings with DME per day day 1 dropped')
plt.xlabel('Days in milk')
plt.ylabel('Proportion of DME')
plt.show()

In [0]:
colors = {1: '#0072B2', 2: '#E69F00', 3: '#009E73'}  # Colorblind-friendly palette (blue, orange, green)

plt.figure(figsize=(10, 6))
for lac_cat in sorted(data_305['LactationCat'].unique()):
    subset_lac = data_305[data_305['LactationCat'] == lac_cat]
    grouped = subset_lac.groupby('DaysInMilk')[['BIM_sessions', 'Valid_sessions']].sum()
    prop_DME_per_day = grouped['BIM_sessions'] / grouped['Valid_sessions']
    if lac_cat == 3: 
        plt.plot(prop_DME_per_day.index, prop_DME_per_day.values, label=f'Parity {lac_cat}+', color=colors.get(lac_cat, None))
    else: 
        plt.plot(prop_DME_per_day.index, prop_DME_per_day.values, label=f'Parity {lac_cat}', color=colors.get(lac_cat, None))

plt.title('Proportion of milkings with DME per day by parity')
plt.xlabel('Days in milk')
plt.ylabel('Proportion of DME')
plt.legend()
plt.show()

In [0]:
#plot the distribution of valid session per days in milk 
grouped = data.groupby('DaysInMilk')['Valid_sessions'].sum()

grouped.plot()
plt.title('Number of valid milkings per day')
plt.xlabel('Days in milk')
plt.ylabel('Number of milkings')
plt.show()

#print number of milking per day for DaysInMilk 1 to 5
print(grouped.loc[0:10])